In [102]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

%precision 3
%matplotlib inline
pd.set_option('display.float_format', '{:.3f}'.format)

In [2]:
#아래는 한글을 사용할 때 깨지는 문제에 대한 해결
from matplotlib import font_manager, rc
font_name = font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
rc('font', family=font_name)

#그래프의 축 등에서 음수를 표시할 때 minus sign이 깨지는 것 해결
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False

In [118]:
df = pd.read_csv('../통계분석2/받은 파일/data/mapo_delivery_3months_50k_dirty.csv')
df

,order_id,order_time,pickup_time,delivery_time,store_dong,customer_dong,distance_km,store_type,weather,temp_c,precipitation_mm,wind_mps,rider_id,time_slot,dow,delivery_minutes
0,O000001,2025.10.24 15:46:54,2025-10-24 16:03:49,10/24 04:07 PM,아현동,신수동,1.170,치킨,Sunny,5.100,0.000,1.200,R0062,기타,4,20.900
1,O000002,2025-12-25 06:44:42,2025-12-25 07:00:15,2025-12-25 07:07:36,망원동,도화동,0.200,야식,비,10.900,4.900,4.600,R0362,기타,3,NaN
2,O000003,2025.12.14 12:52:27,2025-12-14 13:07:15,12/14/2025 13:14,용강동,신촌동,0.360,패스트푸드,비,6.300,2.000,5.100,R0223,점심,6,21.900
3,O000004,2025-11-25 02:58,2025-11-25 03:11:05,2025-11-25 03:40:26,공덕동,서교동,3.590,한식,흐림,5.400,0.000,0.800,R0722,심야,1,41.800
4,O000005,2025-11-24 14:18:45,2025-11-24 14:30:10,NaN,성산동,대흥동,2.540,패스트푸드,맑음,8.600,0.000,5.000,R0128,기타,0,44.300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,O049996,2025-10-21 23:33:36,2025-10-21 23:53:37,2025-10-22 00:21:00,용강동,합정동,3.390,피자,비,8.900,9.500,4.500,R0645,야식,1,47.400
49996,O049997,2025-12-11 22:18:25,12/11/2025 22:35,2025-12-11 22:47:07,신촌동,성산동,1.870,한식,흐림,10.500,0.000,2.000,R0088,야식,3,28.700
49997,O049998,2025-11-12 20:26:28,2025-11-12 20:51:29,2025-11-12 21:06:22,대흥동,망원동,2.760,야식,흐림,5.200,0.000,4.100,R0041,저녁,2,39.900
49998,O049999,2025-12-19 00:22,2025-12-19 00:31:49,2025-12-19 00:48:28,연남동,용강동,1.740,카페/디저트,흐림,3.000,0.000,3.200,R0716,심야,4,25.600


In [5]:
print(type(df['order_id'][0]))
print(type(df['order_time'][0]))
print(type(df['pickup_time'][0]))
print(type(df['delivery_time'][0]))
print(type(df['store_dong'][0]))
print(type(df['customer_dong'][0]))
print(type(df['distance_km'][0]))
print(type(df['store_type'][0]))
print(type(df['weather'][0]))
print(type(df['temp_c'][0]))
print(type(df['precipitation_mm'][0]))
print(type(df['wind_mps'][0]))
print(type(df['rider_id'][0]))
print(type(df['time_slot'][0]))
print(type(df['dow'][0]))
print(type(df['delivery_minutes'][0]))

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'numpy.float64'>
<class 'str'>
<class 'str'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'str'>
<class 'str'>
<class 'numpy.int64'>
<class 'numpy.float64'>


In [67]:
df['order_id'].unique() #106개 중복

array(['O000001', 'O000002', 'O000003', ..., 'O049998', 'O049999',
       'O050000'], shape=(49894,), dtype=object)

In [120]:
# dayfirst=False (월/일/연 처리), yearfirst=True (연/월/일 처리) 등 옵션 조절 가능
# errors='coerce'는 변환 불가한 데이터를 NaT(결측치)로 처리합니다.
lst = ['order_time', 'pickup_time', 'delivery_time']
for i in lst:
    temp_dt = pd.to_datetime(df[i], yearfirst=True, errors='coerce')
    prefix = i.replace('_time', '') # 'order', 'pickup' 등 접두어 추출
    
    df[f'{prefix}_year']  = temp_dt.dt.year.astype('Int64')
    df[f'{prefix}_month'] = temp_dt.dt.month.astype('Int64')
    df[f'{prefix}_day']   = temp_dt.dt.day.astype('Int64')
    df[f'{prefix}_t']     = temp_dt.dt.strftime('%H:%M:%S')

df2 = pd.DataFrame(df[['order_year','order_month','order_day','order_t','pickup_year','pickup_month','pickup_day','pickup_t','delivery_year','delivery_month','delivery_day','delivery_t']])
cols = [c for c in df2.columns if '_t' not in c] # 시간(_t) 제외한 연, 월, 일 컬럼들
for col in cols:
    df2[col] = df2[col].astype('Int64')

for unit in ['year', 'month', 'day', 't']:
    target = f'order_{unit}'
    p_src = f'pickup_{unit}'
    d_src = f'delivery_{unit}'
    
    # order가 NaN이면 pickup 값을, pickup도 NaN이면 delivery 값을 채움
    df2[target] = df2[target].fillna(df2[p_src]).fillna(df2[d_src])
    # [역방향] 혹시 pickup이나 delivery가 비었을 경우를 위해 order 정보를 역으로 전달
    df2[p_src] = df2[p_src].fillna(df2[target])
    df2[d_src] = df2[d_src].fillna(df2[target])

df2.to_csv('tempdata.csv', index = False, encoding = 'utf-8')
df2.head(15)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_20124\2974872608.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_dt = pd.to_datetime(df[i], yearfirst=True, errors='coerce')


,order_year,order_month,order_day,order_t,pickup_year,pickup_month,pickup_day,pickup_t,delivery_year,delivery_month,delivery_day,delivery_t
0,2025,10,24,15:46:54,2025,10,24,16:03:49,2025,10,24,15:46:54
1,2025,12,25,07:00:15,2025,12,25,07:00:15,2025,12,25,07:07:36
2,2025,12,14,12:52:27,2025,12,14,13:07:15,2025,12,14,13:14:00
3,2025,11,25,03:11:05,2025,11,25,03:11:05,2025,11,25,03:40:26
4,2025,11,24,14:30:10,2025,11,24,14:30:10,2025,11,24,14:30:10
5,2026,1,1,21:50:55,2026,1,1,21:50:55,2026,1,1,21:50:55
6,2025,10,24,08:54:58,2025,10,24,08:54:58,2025,10,24,09:19:26
7,2025,12,18,09:34:33,2025,12,18,09:34:33,2025,12,18,09:55:47
8,2025,11,3,18:32:17,2025,11,3,18:32:17,2025,11,3,18:50:20
9,2025,10,25,02:51:46,2025,10,25,02:51:46,2025,10,25,02:51:46
